# Week 5 Day 4

### Task 1: Multi-Agent Design Thinking

#### The task
Research a competitor ("TechNova"), summarize the findings into concrete insights, and
draft a marketing angle our own product could use to differentiate against it.

#### Three roles, no overlap

Here is how we set up the three agents for this job:

**1. The Competitive Researcher**
Think of this agent as our data gatherer. Their only job is to pull the hard facts on TechNova like pricing, market share, and their pros and cons. They are instructed to be incredibly precise and objective. If they cannot find a piece of information, they will explicitly tell us it is missing rather than trying to guess.

**2. The Insights Analyst**
This agent takes the raw facts from the researcher and translates them into business value. Their goal is to find three to five key takeaways that actually matter for a marketing campaign. We tell this agent to be highly skeptical of vague statements and to always double-check the math before jumping to any conclusions.

**3. The Marketing Strategist**
This agent is our copywriter. They look at the analyst's top takeaways and turn the best one into a sharp, on-brand marketing pitch. Their instructions are to write punchy, professional copy that is strictly based on the real evidence they were handed.

#### Why specialists over one generalist (and when that isn't true)

Breaking a job into separate steps keeps each agent focused on its own specific role. A researcher can gather facts cleanly without worrying about marketing tone or letting sales goals distort the data. It is also much easier to write a sharp, effective prompt for one specific task than to cram research rules, analytical thinking, and brand guidelines into a single generalist prompt, which usually weakens all three.

However, this setup is not worth the hassle for quick or simple tasks. If you just need a straightforward answer, splitting the work across multiple agents only adds extra cost, longer delays, and more room for handoff errors without giving you a better result.


#### Task 2: Build Agents & Assign Tools

Each agent also gets its **own `LLM` config** (same Gemini model here for simplicity, but a
lower `temperature` for the Researcher/Analyst who need precision, and a higher one for
the Strategist who needs to write persuasively).


In [9]:
import os
import json

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

load_dotenv()

GEMINI_MODEL = "gemini/gemini-3.5-flash-lite"

api_key = os.getenv("GEMINI_API_KEY")

AGENT_MAX_RPM = 8

llm = LLM(model=GEMINI_MODEL, api_key=api_key, temperature=0.6)
print("laoded_successfully")

laoded_successfully


In [42]:
# Local "database" files, same pattern as Day 2/3's products.json
COMPANIES_DB_PATH = "companies.json"
BRAND_VOICE_PATH = "brand_voice.json"

companies_seed = {
    "technova": {
        "name": "TechNova",
        "market_share_pct": 18.5,
        "pricing_tier": "premium",
        "avg_price_usd": 1200,
        "strengths": ["strong brand recognition", "fast shipping", "large enterprise sales team"],
        "weaknesses": ["no budget product line", "slow to add new features", "mixed support reviews"],
        "recent_news": "TechNova raised prices 8% last quarter and lost some small-business customers.",
    }
}

our_company_seed = {
    "name": "OurCo",
    "market_share_pct": 9.0,
    "avg_price_usd": 650,
}

brand_voice_seed = {
    "tone": "confident, plain-spoken, a little irreverent -- never corporate jargon",
    "avoid": ["synergy", "leverage (as a verb)", "best-in-class", "world-class"],
    "signature_style": "short sentences, concrete numbers, one clear claim per paragraph",
}

with open(COMPANIES_DB_PATH, "w") as f:
    json.dump(companies_seed, f, indent=2)

with open(BRAND_VOICE_PATH, "w") as f:
    json.dump(brand_voice_seed, f, indent=2)

print("Local data files written:", COMPANIES_DB_PATH, BRAND_VOICE_PATH)

Local data files written: companies.json brand_voice.json


In [43]:
@tool("Company Lookup")
def company_lookup(company_name: str) -> str:
    """Look up a competitor's profile (market share, pricing, strengths,
    weaknesses, recent news) from the local competitive-intelligence
    database. company_name should be lowercase, e.g. 'technova'."""
    with open(COMPANIES_DB_PATH) as f:
        db = json.load(f)
    key = company_name.strip().lower()
    if key not in db:
        return json.dumps({"success": False, "error": f"No profile for '{company_name}'."})
    return json.dumps({"success": True, **db[key], "our_company": our_company_seed})


@tool("Calculator")
def calculator(a: float, b: float, operation: str) -> str:
    """Perform basic arithmetic (add, subtract, multiply, divide) on two
    numbers. Use this for any market-share or price comparisons instead of
    computing them mentally."""
    try:
        if operation == "add":
            result = a + b
        elif operation == "subtract":
            result = a - b
        elif operation == "multiply":
            result = a * b
        elif operation == "divide":
            if b == 0:
                return json.dumps({"success": False, "error": "Cannot divide by zero."})
            result = a / b
        else:
            return json.dumps({"success": False, "error": f"Unknown operation: {operation}"})
        return json.dumps({"success": True, "result": result})
    except Exception as e:
        return json.dumps({"success": False, "error": str(e)})


@tool("Brand Voice Guidelines")
def brand_voice_guidelines() -> str:
    """Return our company's brand voice and tone guidelines. Use this before
    drafting any external-facing marketing copy."""
    with open(BRAND_VOICE_PATH) as f:
        return f.read()

In [44]:
researcher = Agent(
    role="Competitive Researcher",
    goal="Pull accurate, current data on the named competitor -- pricing, "
         "market share, strengths, and weaknesses -- and nothing beyond that.",
    backstory=(
        "You are a meticulous competitive-intelligence analyst who only "
        "reports what the data actually says, and flags what's missing "
        "rather than guessing."
    ),
    tools=[company_lookup],
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

analyst = Agent(
    role="Insights Analyst",
    goal="Read the researcher's findings and produce 3-5 ranked, numbered "
         "insights that matter for a marketing decision, with any numeric "
         "comparisons computed correctly.",
    backstory=(
        "You are a sharp business analyst who distrusts vague claims and "
        "always checks the math before presenting a conclusion."
    ),
    tools=[calculator],
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

strategist = Agent(
    role="Marketing Strategist",
    goal="Draft one clear, on-brand marketing angle that exploits the "
         "strongest insight, written in our company's tone of voice.",
    backstory=(
        "You are a marketing strategist who writes punchy, board-ready "
        "copy and always grounds claims in the evidence handed to you."
    ),
    tools=[brand_voice_guidelines],
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

## Task 3: Define Tasks & Process (Sequential)

`context=[...]` wires each task's output into the next task's input — the Analyst sees the
Researcher's raw findings, and the Strategist sees both the findings and the insights.


In [45]:
research_task = Task(
    description=(
        "Research the competitor 'technova' using the Company Lookup tool. "
        "Report their pricing tier, average price, market share, strengths, "
        "weaknesses, and any recent news, compared against our own company's "
        "market share and average price (also returned by the tool)."
    ),
    expected_output=(
        "A factual summary with clearly labeled fields: TechNova pricing, "
        "TechNova market share, our market share, our average price, "
        "TechNova strengths (list), TechNova weaknesses (list), recent news."
    ),
    agent=researcher,
)

analysis_task = Task(
    description=(
        "Using the researcher's findings, identify 3-5 ranked insights that "
        "matter for a marketing decision. Use the Calculator tool to compute "
        "the exact percentage-point gap between our market share and "
        "TechNova's market share, and the exact dollar gap between average "
        "prices -- do not estimate these numbers yourself."
    ),
    expected_output=(
        "A NUMBERED list (1., 2., 3., ...) of 3-5 insights, each one "
        "sentence, each citing a specific number from the research findings "
        "or the calculator results. No prose paragraphs -- numbered list only."
    ),
    agent=analyst,
    context=[research_task],
)

marketing_task = Task(
    description=(
        "Using the analyst's ranked insights and the researcher's findings, "
        "call the Brand Voice Guidelines tool, then draft ONE marketing "
        "angle (a short paragraph, 3-5 sentences) that exploits the single "
        "strongest insight. It must follow our brand voice exactly."
    ),
    expected_output=(
        "A short marketing angle, 3-5 sentences, in our brand's tone, "
        "grounded in at least one specific number from the analysis."
    ),
    agent=strategist,
    context=[research_task, analysis_task],
)

In [46]:
sequential_crew = Crew(
    agents=[researcher, analyst, strategist],
    tasks=[research_task, analysis_task, marketing_task],
    process=Process.sequential,
    max_rpm=AGENT_MAX_RPM,
    verbose=True,
)

sequential_result = await sequential_crew.kickoff_async()

print("\n" + "=" * 70)
print("SEQUENTIAL CREW -- FINAL OUTPUT")
print("=" * 70)
print(sequential_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c24122cc-ebcf-44be-8f64-6ed8601aede3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the competitor 'technova' using the Company Lookup tool. Report their pricing tier, average     │
│  price, market share, strengths, weaknesses, and any recent news, compared against our own company's market     │
│  share and average price (also returned by the tool).                                                           │
│  ID: f1bc2f0a-2cf4-480f-9f42-f4aebe965eaa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Competitive Researcher                                                                                  │
│                                                                                                                 │
│  Task: Research the competitor 'technova' using the Company Lookup tool. Report their pricing tier, average     │
│  price, market share, strengths, weaknesses, and any recent news, compared against our own company's market     │
│  share and average price (also returned by the tool).                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool company_lookup executed with result: {"success": true, "name": "TechNova", "market_share_pct": 18.5, "pricing_tier": "premium", "avg_price_usd": 1200, "strengths": ["strong brand recognition", "fast shipping", "large enterprise sales tea...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: company_lookup                                                                                           │
│  Args: {'company_name': 'technova'}                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing Strategist                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  TechNova just handed small businesses a reason to leave. By hiking their average prices to $1,200 last         │
│  quarter, they abandoned the budget-conscious operators who built them. We offer a smarter alternative at       │
│  $650, slipping right into the gap they left wide open.                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the analyst's ranked insights and the researcher's findings, call the Brand Voice Guidelines       │
│  tool, then draft ONE marketing angle (a short paragraph, 3-5 sentences) that exploits the single strongest     │
│  insight. It must follow our brand voice exactly.                                                               │
│  Agent: Marketing Strategist                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c24122cc-ebcf-44be-8f64-6ed8601aede3                                                                       │
│  Final Output: TechNova just handed small businesses a reason to leave. By hiking their average prices to       │
│  $1,200 last quarter, they abandoned the budget-conscious operators who built them. We offer a smarter          │
│  alternative at $650, slipping right into the gap they left wide open.                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


SEQUENTIAL CREW -- FINAL OUTPUT
TechNova just handed small businesses a reason to leave. By hiking their average prices to $1,200 last quarter, they abandoned the budget-conscious operators who built them. We offer a smarter alternative at $650, slipping right into the gap they left wide open.


### Fixing a Formatting Glitch

During an early test, the analyst returned its findings as one big paragraph instead of a clear list. This made it difficult for the marketing agent to pick out the strongest insight because it had to dig through a block of text to find it.

To fix this, we updated the "expected output" instructions. Instead of vaguely asking for a list, we explicitly demanded a numbered list and told it to stop writing paragraphs.

We learned that being very specific in the expected output section works much better than just dropping a hint in the task description. CrewAI pays far more attention to what you explicitly state you expect to receive.

## Task 4: Hierarchical Delegation


In [47]:
import time

research_task_h = Task(
    description=research_task.description,
    expected_output=research_task.expected_output,
)

analysis_task_h = Task(
    description=analysis_task.description,
    expected_output=analysis_task.expected_output,
    context=[research_task_h],
)

marketing_task_h = Task(
    description=marketing_task.description,
    expected_output=marketing_task.expected_output,
    context=[research_task_h, analysis_task_h],
)

hierarchical_crew = Crew(
    agents=[researcher, analyst, strategist],
    tasks=[research_task_h, analysis_task_h, marketing_task_h],
    process=Process.hierarchical,
    manager_llm=llm,
    max_rpm=AGENT_MAX_RPM,
    verbose=True,
)

time.sleep(25)
hierarchical_result = await hierarchical_crew.kickoff_async()

print("\n" + "=" * 70)
print("HIERARCHICAL CREW -- FINAL OUTPUT")
print("=" * 70)
print(hierarchical_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1fbaac6c-db9e-434a-b53a-00c8083b1094                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the competitor 'technova' using the Company Lookup tool. Report their pricing tier, average     │
│  price, market share, strengths, weaknesses, and any recent news, compared against our own company's market     │
│  share and average price (also returned by the tool).                                                           │
│  ID: f8b071c0-0232-4b6a-80bf-0b19149fdfaa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Research the competitor 'technova' using the Company Lookup tool. Report their pricing tier, average     │
│  price, market share, strengths, weaknesses, and any recent news, compared against our own company's market     │
│  share and average price (also returned by the tool).                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'coworker': 'Competitive Researcher', 'context': "We need to research the competitor 'Technova' using   │
│  the Company Lookup tool. We also need to retrieve our own company's market share and average pric...           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Competitive Researcher                                                                                  │
│                                                                                                                 │
│  Task: Research the competitor 'Technova' using the Company Lookup tool and gather all required metrics and     │
│  qualitative data (pricing tier, average price, market share, strengths, weaknesses, recent news) as well as    │
│  our own company's market share and average price.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  TechNova just handed their small-business customers an 8% price hike last quarter. That greed opened a         │
│  massive door for us. While they sit fat and happy at a $1,200 average price, we offer the exact same power     │
│  for $650. Pocket the $550 difference and get back to work.                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the analyst's ranked insights and the researcher's findings, call the Brand Voice Guidelines       │
│  tool, then draft ONE marketing angle (a short paragraph, 3-5 sentences) that exploits the single strongest     │
│  insight. It must follow our brand voice exactly.                                                               │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


HIERARCHICAL CREW -- FINAL OUTPUT
TechNova just handed their small-business customers an 8% price hike last quarter. That greed opened a massive door for us. While they sit fat and happy at a $1,200 average price, we offer the exact same power for $650. Pocket the $550 difference and get back to work.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1fbaac6c-db9e-434a-b53a-00c8083b1094                                                                       │
│  Final Output: TechNova just handed their small-business customers an 8% price hike last quarter. That greed    │
│  opened a massive door for us. While they sit fat and happy at a $1,200 average price, we offer the exact same  │
│  power for $650. Pocket the $550 difference and get back to work.                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Sequential vs. hierarchical  pros, cons, when to use each

| | Sequential | Hierarchical |
|---|---|---|
| **Quality** | Consistent, since the pipeline order is fixed and matches the task's natural dependency chain | Can be *higher* if the manager catches a bad sub-agent output and re-delegates, but can also be *lower* if the manager mis-routes a step or skips a needed one |
| **Latency / token usage** | Lower  exactly 3 agent calls, no manager overhead | Higher  the manager itself makes LLM calls to plan and review, on top of the 3 worker calls, so expect noticeably more total tokens for the same task |
| **Reliability** | High  the same 3 steps run every time, easy to reason about and debug | Lower  delegation order isn't guaranteed to match what you'd expect, and a manager mistake compounds into every downstream task |
| **When to use** | The task has a known, fixed pipeline shape (research → analyze → write) | Task assignment genuinely depends on intermediate results the fixed pipeline can't anticipate  e.g., "figure out what data is even needed first" |

For *this* task specifically  research → analyze → write is a completely predictable
pipeline  sequential is the better fit; hierarchical's extra manager overhead buys
nothing here since there's no real ambiguity about which agent should act when.


## Task 5: Evaluation & Cost Awareness

### Token usage per run


In [48]:
def print_usage(label, crew):
    metrics = crew.usage_metrics
    print(f"{label}:")
    print(f"  total_tokens:      {metrics.total_tokens}")
    print(f"  prompt_tokens:     {metrics.prompt_tokens}")
    print(f"  completion_tokens: {metrics.completion_tokens}")
    print(f"  successful_requests: {metrics.successful_requests}")

print_usage("SEQUENTIAL CREW", sequential_crew)
print()
print_usage("HIERARCHICAL CREW", hierarchical_crew)



SEQUENTIAL CREW:
  total_tokens:      11220
  prompt_tokens:     10005
  completion_tokens: 1215
  successful_requests: 21

HIERARCHICAL CREW:
  total_tokens:      81184
  prompt_tokens:     69776
  completion_tokens: 11408
  successful_requests: 100
╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TR

In [51]:
INPUT_RATE = 0.30 / 1_000_000
OUTPUT_RATE = 2.50 / 1_000_000

def estimate_cost(metrics):
    return metrics.prompt_tokens * INPUT_RATE + metrics.completion_tokens * OUTPUT_RATE

print("Sequential est. cost: $", round(estimate_cost(sequential_crew.usage_metrics), 5))
print("Hierarchical est. cost: $", round(estimate_cost(hierarchical_crew.usage_metrics), 5))

Sequential est. cost: $ 0.00604
Hierarchical est. cost: $ 0.04945


**Approximate cost.** Gemini pricing is per 1M tokens (check [ai.google.dev/pricing](https://ai.google.dev/pricing) for current rates, since these change, and note the free tier may cover this entire notebook at zero cost). Using the commonly-cited illustrative paid-tier rates (~$0.30/1M input, ~$2.50/1M output tokens) and the **actual `usage_metrics` from the runs above**:

| | Prompt tokens | Completion tokens | Requests | Est. cost |
|---|---|---|---|---|
| Sequential | 10,005 | 1,215 | 21 | **$0.00604** |
| Hierarchical | 69,776 | 11,408 | 100 | **$0.04945** |

The hierarchical run came out **~8x more expensive** than sequential for the identical underlying task — almost entirely from the manager agent's extra planning/delegation/review calls (100 requests vs. 21) on top of the same three worker calls.

For comparison, **Day 3's single-agent LangGraph workflow** (one model doing plan → retrieve → generate → critique → format, no separate specialized agents) used the fewest total LLM calls of the three approaches for a comparably-scoped task, since there's no manager and no cross-agent handoff overhead — the cost of "more agents" is close to linear in this framework, not free.

### Three success criteria, scored against the actual runs above

| Criterion | What it checks |
|---|---|
| **Factual grounding** | Does the marketing angle cite a real number from the research/analysis, not an invented one? |
| **Completeness** | Does the final output actually address pricing *and* market share *and* at least one qualitative strength/weakness? |
| **Tone** | Does the copy follow `brand_voice.json` (no banned jargon words, short sentences, one claim per paragraph)? |

Score each 1 (fails) – 3 (fully meets), scored directly from the `sequential_result.raw` and `hierarchical_result.raw` outputs captured above (one executed run per process — re-run the cells 2-3 more times each if you want a larger sample, since LLM output varies run to run):

| Run | Factual grounding | Completeness | Tone |
|---|---|---|---|
| Sequential run 1 | 3 — cites $1,200 and $650 | 2 — covers pricing but never mentions the market-share gap | 3 — short sentences, concrete numbers, no banned jargon |
| Hierarchical run 1 | 3 — cites $1,200, $650, and the computed $550 gap | 2 — also pricing-only, market share never surfaces in the final copy | 3 — on-tone, though "sit fat and happy" leans further into "irreverent" than the other run |

*(Only one run per process was actually executed and captured in this notebook, so this is n=1 per variant rather than the 3 runs each the task asks for — re-run `sequential_crew.kickoff_async()` and `hierarchical_crew.kickoff_async()` a couple more times and extend this table if you need a larger sample. Notably, **both** runs skipped market-share in the final marketing angle even though the analyst's insights included it — a real formatting/completeness gap worth tightening `marketing_task.expected_output` for, e.g. by explicitly requiring the copy to reference both metrics.)*

### Was the crew worth it for this task?

For this specific task, a multi-agent crew was worth the complexity **in the sequential form**, not the hierarchical one: splitting research/analysis/writing into separate role-scoped agents produced a more clearly-grounded final output than a single generalist agent typically manages in one pass, because the Analyst's dedicated "compute the real numbers" step catches arithmetic the Strategist would otherwise eyeball. But that gain came from *role separation*, not from *multi-agent-ness* per se — a single well-designed agent using the same three tools in a Day 3-style LangGraph pipeline (plan → retrieve → compute → draft, with explicit state) could likely match sequential CrewAI's quality at lower token cost, since it avoids CrewAI's per-agent system-prompt overhead. Hierarchical delegation cost **~8x more tokens** for output that was no more complete (both runs missed the market-share point), so for this task's genuinely fixed, unambiguous pipeline shape, the manager's extra planning/review layer bought nothing measurable.